# Clase 1 · El precio de la luz (gemelo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelve cada uno, de forma individual, en la segunda mitad de la clase. Es el gemelo de *El sancocho*: la misma técnica en otro mundo.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Repetir por cuenta propia el camino de la clase: formular, limpiar, vencer una línea base con validación temporal y convertir el resultado en una decisión. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular), 2 (seleccionar y preparar datos), 3 (flujo reproducible), 4 (implementar y evaluar) y 5 (comunicar). |
| **Uso de IA** | Pueden pedirle a Claude que explique una celda, que les ayude a llenar un hueco o que interprete un error. Deben poder explicar sin ayuda la regla con que se escogió la versión del precio, por qué el corte es temporal y qué significa su MAPE. |
| **Entregable** | El cuaderno completo y corrido en su portafolio de GitHub, con las predicciones escritas, la decisión en tres frases y la bitácora llenas. |

**Cómo se trabaja.** El código está escrito casi todo. Donde vean `____` falta una pieza, y el comentario de al lado dice qué va. Lean la celda entera antes de llenar. Si una celda falla, lean el error antes de pedir ayuda.

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

## 1. Formulación

En Colombia la energía se compra y se vende cada hora en un mercado que administra XM. El precio de ese mercado se llama **precio de bolsa** y se mide en pesos por kilovatio hora. La explicación que uno oye siempre es que el precio sube cuando los embalses bajan, porque hay que prender plantas térmicas, que son más caras. Hoy vamos a ver cuánto de esa explicación aguanta el dato.

Imagínense una empresa mediana de Manizales que gasta mucha energía. Puede comprarla al precio de bolsa, que cambia todos los días, o firmar un precio fijo. Su gerente necesita tres respuestas:

1. **Predecir.** ¿Cuál será el precio de bolsa promedio dentro de siete días?
2. **Explicar.** ¿Cuánto del precio se entiende mirando el nivel de los embalses?
3. **Decidir.** Si le ofrecen un precio fijo para la semana entrante, ¿desde qué valor le conviene firmar?

**La métrica** es la misma de la clase: el MAPE del pronóstico, ahora a siete días.

### Predicción escrita

Escriban su apuesta antes de correr nada.

- ¿Cuánto vale hoy un kilovatio hora en bolsa, en pesos? **Mi apuesta:**
- ¿Qué tan lejos queda el precio de dentro de siete días del precio de hoy, en porcentaje? **Mi apuesta:**
- Cuando los embalses están llenos, ¿el precio es alto o bajo? **Mi apuesta:**

## 2. Datos

Pedimos dos tablas a XM: el precio de bolsa hora por hora y la energía guardada en cada embalse día por día, de enero de 2023 a octubre de 2026. **La descarga tarda entre cinco y seis minutos** porque XM entrega un mes por consulta. Córranla y lean la sección mientras termina.

In [ ]:
precio_crudo = descargar("xm_precio")
embalses_crudo = descargar("xm_embalses")
precio_crudo.head(8)

### 2.1 La primera trampa: cada hora aparece muchas veces

Miren la tabla de arriba. La misma hora sale repetida. Averigüemos por qué.

In [ ]:
print("Filas:", len(precio_crudo), "| Horas distintas:", precio_crudo["FechaHora"].nunique())
print(precio_crudo.groupby(["CodigoVariable", "Version"]).size().unstack())

Hay tres variables y hasta diez versiones:

- `PB_Nal` es el precio de bolsa nacional. `PB_Int` y `PB_Tie` son precios para intercambios con otros países. A la empresa le interesa el nacional.
- XM liquida cada hora varias veces. Las versiones numeradas van de `TX1` a `TX8`: casi todas las horas tienen `TX1` y muy pocas llegan a `TX8`, así que el número indica cuántas veces se ha vuelto a liquidar esa hora. Además están `TXF` y `TXR`, que existen para todas las horas.

Si promediaran la tabla tal como viene, mezclarían precios internacionales con nacionales y contarían la misma hora varias veces. Además, las versiones no siempre coinciden. Miren un día concreto:

In [ ]:
un_dia = precio_crudo[(precio_crudo["CodigoVariable"] == "PB_Nal") & precio_crudo["FechaHora"].str.startswith("2024-09-30")]
print(un_dia.groupby("Version")["Valor"].mean().round(0))

El mismo día vale más de 7.000 pesos según unas versiones y menos de 2.500 según otras. Quien escoja otra versión estará modelando otro precio.

**Regla que vamos a usar:** nos quedamos con el precio nacional y, para cada hora, con la liquidación numerada más reciente que exista (`TX8` si la hay, si no `TX7`, y así hasta `TX1`). Solo si una hora no tiene ninguna numerada usamos `TXF`. La razón es que una liquidación posterior corrige a las anteriores.

Esta regla es una decisión de modelación que tomé mirando el dato, sin tener a la mano la definición oficial de cada sigla. Anótenla en la bitácora y, en la sección de riesgos, digan qué habría que consultarle a XM.

In [ ]:
nacional = precio_crudo[precio_crudo["CodigoVariable"] == ____].copy()   # HUECO 1: el código del precio nacional, entre comillas

preferencia = ["TX8", "TX7", "TX6", "TX5", "TX4", "TX3", "TX2", "TX1", "TXF", "TXR"]
nacional["orden"] = nacional["Version"].map({v: i for i, v in enumerate(preferencia)})
horario = (nacional.sort_values(["FechaHora", "orden"])
           .drop_duplicates("FechaHora", keep=____))                       # HUECO 2: "first" o "last". ¿Cuál deja la versión preferida?

print("Horas después de limpiar:", len(horario), "| ¿Alguna hora repetida?", horario["FechaHora"].duplicated().any())
print(horario["Version"].value_counts().sort_index())

**Compruébenlo ustedes mismos.** El número de horas después de limpiar debe ser igual al número de horas distintas que imprimieron antes. Si no coincide, revisen los huecos 1 y 2.

### 2.2 Del precio horario al precio diario

In [ ]:
horario["fecha"] = pd.to_datetime(horario["FechaHora"]).dt.normalize()
precio = horario.groupby("fecha")["Valor"].____().rename("precio")           # HUECO 3: la operación que resume las 24 horas en un promedio

ax = precio.plot(color="#16302B", lw=1)
ax.set(title="Precio de bolsa nacional, promedio diario", xlabel="", ylabel="COP/kWh"); plt.show()
print(precio.describe().round(1))

Comparen con su primera apuesta. ¿Qué pasó entre finales de 2023 y abril de 2024? Escríbanlo en una línea: **Mi explicación:**

### 2.3 El nivel de los embalses

La tabla trae, por embalse y por día, la energía útil guardada y la capacidad útil. El nivel agregado del país es la suma de lo guardado sobre la suma de la capacidad.

**Segunda trampa: filas que ya son sumas.** Miren los nombres de los embalses.

In [ ]:
print(sorted(embalses_crudo["CodigoEmbalse"].unique()))

Tres nombres no son embalses. `AGREGADO_SIN` es el total del país, que XM publica desde julio de 2025. `AGREGADO_BOGOTA`, que desde enero de 2025 aparece como `AGREGADO`, es la suma de Neusa, Sisga y Tominé, que hasta junio de 2025 también vienen por separado. Si sumaran todas las filas contarían esa agua dos y hasta tres veces.

La celda siguiente quita las filas repetidas y calcula el nivel. Después lo compara con el total que publica XM, para saber si la regla quedó bien.

In [ ]:
embalses_crudo["fecha"] = pd.to_datetime(embalses_crudo["Fecha"])
repetidas = ["AGREGADO_SIN", "NEUSA", "SISGA", "TOMINE"]
sin_repetir = embalses_crudo[~embalses_crudo["CodigoEmbalse"].isin(repetidas)]
por_dia = sin_repetir.groupby("fecha")[["VolumenUtilDiarioEnergia", "CapacidadUtilEnergia"]].sum()
embalse = (por_dia["VolumenUtilDiarioEnergia"] / por_dia[____] * 100).rename("embalse")   # HUECO 4: la columna que va en el denominador

oficial = embalses_crudo[embalses_crudo["CodigoEmbalse"] == "AGREGADO_SIN"].set_index("fecha")
oficial = oficial["VolumenUtilDiarioEnergia"] / oficial["CapacidadUtilEnergia"] * 100
diferencia = (embalse.reindex(oficial.index) - oficial).abs().max()
print(f"Diferencia máxima con el total de XM, en {len(oficial)} días comparables: {diferencia:.3f} puntos")

datos = pd.concat([precio, embalse], axis=1).dropna()
print(len(datos), "días, de", datos.index.min().date(), "a", datos.index.max().date())
datos["embalse"].plot(color="#3E7CB1"); plt.title("Nivel agregado de los embalses (%)"); plt.xlabel(""); plt.show()

## 3. Modelado

### 3.1 La línea base

Dos pronósticos que no cuestan nada:

- **Ingenuo:** dentro de siete días el precio será el de hoy.
- **Promedio semanal:** dentro de siete días el precio será el promedio de los últimos siete días.

In [ ]:
H = 7                                                         # horizonte: siete días
tabla = pd.DataFrame({"real": datos["precio"].shift(____)})   # HUECO 5: el desplazamiento que trae el precio de dentro de H días a la fila de hoy
tabla["ingenuo"] = datos["precio"]
tabla["promedio_7"] = datos["precio"].rolling(7).mean()
tabla = tabla.dropna()

def mape(real, pronostico):
    return float((abs(real - pronostico) / ____).mean() * 100)   # HUECO 6: ¿contra qué se compara el error para volverlo porcentaje?

for metodo in ["ingenuo", "promedio_7"]:
    print(f"MAPE {metodo}: {mape(tabla['real'], tabla[metodo]):.1f} %")

Comparen con su segunda apuesta.

### 3.2 Un modelo, con validación temporal

La regresión mira el precio de hoy, el promedio de la semana y el nivel de los embalses.

**Predicción escrita:** ¿le ganará la regresión al pronóstico ingenuo? **Mi apuesta:**

In [ ]:
X = pd.DataFrame({
    "hoy": datos["precio"],
    "promedio_7": datos["precio"].rolling(7).mean(),
    "embalse": datos["embalse"],
})
base = X.join(datos["precio"].shift(-H).rename("real")).dropna()
variables = list(X.columns)

corte = "2025-06-30"
entreno = base.loc[:corte]
prueba = base.loc[____:].iloc[1:]                            # HUECO 7: la prueba empieza donde termina el entrenamiento
modelo = LinearRegression().fit(entreno[variables], entreno[____])   # HUECO 8: la columna que el modelo debe aprender a adivinar
pronostico = pd.Series(modelo.predict(prueba[variables]), index=prueba.index)

comparacion = pd.DataFrame({
    "real": prueba["real"],
    "ingenuo": tabla.loc[prueba.index, "ingenuo"],
    "promedio_7": tabla.loc[prueba.index, "promedio_7"],
    "regresion": pronostico,
})
resumen = pd.Series({m: mape(comparacion["real"], comparacion[m]) for m in ["ingenuo", "promedio_7", "regresion"]}).round(1)
print(f"Entrenamiento: {len(entreno)} días | Prueba: {len(prueba)} días")
print("MAPE en la prueba, en %:")
print(resumen.sort_values())

**¿Por qué el corte es una fecha y no un sorteo?** Respondan en dos líneas, con sus palabras: **Mi respuesta:**

In [ ]:
grafica = comparacion.copy(); grafica.index = grafica.index + pd.Timedelta(days=H)
ax = grafica["real"].plot(color="black", lw=1.5, label="real")
grafica["ingenuo"].plot(ax=ax, color="#E0A526", lw=1, label="ingenuo")
grafica["regresion"].plot(ax=ax, color="#A8432C", lw=1, ls="--", label="regresión")
ax.set(title="Pronóstico a siete días contra el precio real", xlabel="", ylabel="COP/kWh"); ax.legend(); plt.show()

### 3.3 Explicar: ¿cuánto del precio se entiende con los embalses?

In [ ]:
correlacion = datos["precio"].corr(datos[____])           # HUECO 9: la otra variable de la relación
print(f"Correlación entre precio y nivel de embalses: {correlacion:.2f}")

plt.scatter(datos["embalse"], datos["precio"], s=6, alpha=0.4, color="#16302B")
plt.xlabel("Nivel agregado de embalses (%)"); plt.ylabel("Precio de bolsa (COP/kWh)")
plt.title("Cada punto es un día"); plt.show()

print("Precio promedio según el nivel de los embalses:")
print(datos.groupby(pd.cut(datos["embalse"], [0, 40, 50, 60, 70, 100]), observed=True)["precio"].agg(["mean", "count"]).round(0))

Comparen con su tercera apuesta. Luego respondan:

- ¿La relación es una línea recta? **Mi lectura:**
- Una correlación no prueba que los embalses causen el precio. ¿Qué otra cosa podría mover las dos variables a la vez? **Mi respuesta:**

## 4. Comunicación

### 4.1 Decidir: ¿desde qué precio fijo conviene firmar?

Con los errores de la prueba armamos una franja alrededor del pronóstico. Usamos el método que ganó en la sección 3.

In [ ]:
mejor = resumen.idxmin()
razon = comparacion["real"] / comparacion[mejor]               # real dividido entre pronosticado, día por día
bajo, alto = razon.quantile(0.10), razon.quantile(____)      # HUECO 10: el percentil que deja por debajo 9 de cada 10 días

hoy = datos.index[-1]
if mejor == "ingenuo":
    punto = datos["precio"].iloc[-1]
elif mejor == "promedio_7":
    punto = datos["precio"].rolling(7).mean().iloc[-1]
else:
    punto = float(modelo.predict(X.iloc[[-1]][variables])[0])

print(f"Método elegido: {mejor} (MAPE {resumen[mejor]} %)")
print(f"Pronóstico para el {(hoy + pd.Timedelta(days=H)).date()}: {punto:.0f} COP/kWh")
print(f"Franja de 8 de cada 10 días: entre {punto * bajo:.0f} y {punto * alto:.0f} COP/kWh")

Lean la franja así. Si a la empresa le ofrecen un precio fijo por debajo del borde inferior, casi siempre le conviene firmar. Si se lo ofrecen por encima del borde superior, casi siempre le conviene la bolsa. En el medio, la decisión depende de cuánto le duele un susto.

### La decisión en tres frases

Para el gerente, que no sabe qué es un percentil.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

Llenen las filas vacías. La primera va de ejemplo.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Precio nacional (`PB_Nal`) | `PB_Int`, `PB_Tie` | La empresa compra en el mercado nacional |
| Versión del precio | | |
| Filas de embalses que se quitaron | | |
| Promedio diario simple | | |
| Corte temporal en junio de 2025 | | |
| Franja con percentiles 10 y 90 | | |

### 4.3 Riesgos y mantenimiento

- Los días más recientes solo tienen las primeras liquidaciones, que después se corrigen. ¿Qué le pasa a un pronóstico que se hace con datos provisionales?
- ¿Qué le preguntarían a XM sobre las versiones `TXF` y `TXR` antes de usar este análisis con plata de verdad?
- El promedio diario simple trata igual la hora de las 3 de la mañana y la de las 7 de la noche. ¿A qué empresa le sirve mal ese promedio?
- Si llega un fenómeno de El Niño, ¿en cuál de sus tres respuestas confiarían menos?

### 4.4 Procedencia y entrega

In [ ]:
procedencia()

**Para entregar:** en Colab, menú *Archivo*, *Guardar una copia en GitHub*, y escojan su repositorio de portafolio, carpeta `clase01/`. Un cuaderno que no corre de principio a fin desde cero no se califica: antes de guardar usen *Entorno de ejecución*, *Reiniciar sesión y ejecutar todo*.